# Re-run: what is identified (Paper II, RQ5)

This notebook recomputes the headline numbers of RQ5 from the committed outputs in the code package of
release `learning-graph-budget-2026`. It needs Python 3.9+ with numpy and the unzipped release (run it
from the release folder or its `notebooks/` folder, or set `RELEASE_DIR`). The refit of the
cross-curricular regressions with their 2,000 joint bootstrap draws takes about half a minute.

**What it recomputes, and from what.**

- **Across four curricula.** For each corpus and pair of Common Core clusters, the release keeps D (the
  partial correlation of learners' accuracy on the two clusters, given the other clusters) and P (local
  teaching proximity), in `core-vs-culture/out/invariance-pairs.json`. Computing D and P needs the
  corpora (XES3G5M, ASSISTments 2012–13, Eedi, Junyi 2020), which may not be redistributed; everything
  after them does not. The notebook refits the 12 regressions of one corpus's standardized D on
  another's and on its own P, pools them into b_other and b_prox, and redraws the joint bootstrap and
  permutation null with the study's seed, so the intervals and the null are recomputed too.
- **The person's check of the map.** The release keeps the coder's code for each sampled unit
  (`human-codes.json`) and both models' maps (`unit-map.json`). The notebook recomputes Cohen's kappa
  between them, by corpus (Table 14), with the bootstrap interval, and the unregistered reading that
  counts the 13 blank items as no cluster.
- **The pair-level scan (Table 13)** from its per-set records: the most negative z, the critical value,
  which sets fire with an isolated pair, the pairs they flag, and the binomial p of 8 firing sets.
- **Table 12** from the replication rows of the low-rank study.

The item response fits, the bootstrap of the scan and the ECPE test read corpora and are not redone.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "learning-graph-budget-2026"   # set when the report is built
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Code package:", shown(code_dir()))

Code package: code


In [2]:
import json, math
from collections import Counter
import numpy as np

sg = lambda x, d=3: f"{x:+.{d}f}".replace("-", "−")
stats = load_code_json("studies/identifiability/core-vs-culture/out/invariance-pairs.json")
committed = load_code_json("studies/identifiability/core-vs-culture/out/invariance-result.json")["openai/gpt-6-sol"]["cross"]
MIN_SHARED, SEED = 30, 20260927          # invariance.py's constants

def zs(v):
    v = np.asarray(v, float)
    sd = v.std()
    return (v - v.mean()) / sd if sd > 0 else v * 0

per = {}
for c in stats:
    for c2 in stats:
        if c != c2:
            shared = sorted(k for k in stats[c] if k in stats[c2] and stats[c][k]["P"] is not None)
            if len(shared) >= MIN_SHARED:
                per[f"{c}<-{c2}"] = shared

def fit(c, c2, keys, dmap=None):
    """One corpus's standardized D on another's D (b_other) and on its own proximity P (b_prox)."""
    d = zs([stats[c][k]["D"] for k in keys])
    src = dmap[c2] if dmap is not None else {k: stats[c2][k]["D"] for k in keys}
    X = np.column_stack([np.ones(len(keys)), zs([src[k] for k in keys]), zs([stats[c][k]["P"] for k in keys])])
    beta = np.linalg.lstsq(X, d, rcond=None)[0]
    return beta[1], beta[2]

point = {k: fit(*k.split("<-"), keys) for k, keys in per.items()}
for k, (o, p) in point.items():
    print(f"{k:28s} shared {len(per[k]):5,d}  b_other {sg(o)}  b_prox {sg(p)}")
b_other = float(np.mean([v[0] for v in point.values()]))
b_prox = float(np.mean([v[1] for v in point.values()]))
print(f"pooled over {len(point)} ordered corpus pairs: b_other {sg(b_other)}, b_prox {sg(b_prox)}")
assert len(point) == 12 and min(len(v) for v in per.values()) == 273 and max(len(v) for v in per.values()) == 1134
assert all(v[0] > 0 for v in point.values())                     # positive in all 12 ordered pairs
assert (sg(min(v[0] for v in point.values())), sg(max(v[0] for v in point.values()))) == ("+0.062", "+0.228")
assert (sg(b_other), sg(b_prox), f"{b_other / b_prox:.3f}") == ("+0.125", "+0.363", "0.344")

xes3g5m<-assistments2012     shared   273  b_other +0.084  b_prox +0.379
xes3g5m<-eedi                shared   395  b_other +0.148  b_prox +0.364
xes3g5m<-junyi2020           shared   804  b_other +0.108  b_prox +0.390
assistments2012<-xes3g5m     shared   273  b_other +0.062  b_prox +0.270
assistments2012<-eedi        shared   760  b_other +0.162  b_prox +0.274
assistments2012<-junyi2020   shared   768  b_other +0.097  b_prox +0.290
eedi<-xes3g5m                shared   395  b_other +0.121  b_prox +0.615
eedi<-assistments2012        shared   760  b_other +0.118  b_prox +0.595
eedi<-junyi2020              shared 1,134  b_other +0.162  b_prox +0.508
junyi2020<-xes3g5m           shared   804  b_other +0.095  b_prox +0.234
junyi2020<-assistments2012   shared   768  b_other +0.114  b_prox +0.237
junyi2020<-eedi              shared 1,134  b_other +0.228  b_prox +0.200
pooled over 12 ordered corpus pairs: b_other +0.125, b_prox +0.363


In [3]:
# The intervals and the null, redrawn exactly as the study draws them: one bootstrap draw of cluster pairs
# from their union serves every regression, and the null shuffles each corpus's D over its own pairs once
# per draw. Same seed, same order of random draws.
rng = np.random.default_rng(SEED)
union = sorted(set().union(*per.values()))
boot_o, boot_p, null = [], [], []
for _ in range(2000):
    cnt = Counter(union[i] for i in rng.integers(0, len(union), len(union)))
    bo, bp = [], []
    for k, keys in per.items():
        c, c2 = k.split("<-")
        sample = [x for x in keys for _ in range(cnt.get(x, 0))]
        if len(sample) >= 3:
            o, p = fit(c, c2, sample)
            bo.append(o)
            bp.append(p)
    boot_o.append(np.mean(bo))
    boot_p.append(np.mean(bp))
    dmap = {}
    for c in stats:
        ks = list(stats[c])
        dmap[c] = dict(zip(ks, rng.permutation([stats[c][x]["D"] for x in ks])))
    null.append(np.mean([fit(*k.split("<-"), keys, dmap=dmap)[0] for k, keys in per.items()]))
ci_o, ci_p = np.percentile(boot_o, [2.5, 97.5]), np.percentile(boot_p, [2.5, 97.5])
null975 = float(np.percentile(null, 97.5))
core = ci_o[0] > 0 and b_other > null975
verdict = "CORE AND CULTURAL" if core and ci_p[0] > 0 else "CORE" if core else "CULTURAL" if ci_p[0] > 0 else "NEITHER"
print(f"b_other {sg(b_other)} [{sg(ci_o[0])}, {sg(ci_o[1])}], null 97.5th percentile {sg(null975)}; "
      f"b_prox {sg(b_prox)} [{sg(ci_p[0])}, {sg(ci_p[1])}] -> {verdict}")
assert (sg(ci_o[0]), sg(ci_o[1]), sg(ci_p[0]), sg(ci_p[1]), sg(null975)) == ("+0.091", "+0.158", "+0.326", "+0.401", "+0.022")
assert abs(ci_o[0] - committed["b_other_ci"][0]) < 1e-12 and abs(null975 - committed["null_b_other_975"]) < 1e-12
assert verdict == committed["verdict"] == "CORE AND CULTURAL"
print("Refitted: b_other +0.125 [+0.091, +0.158] above its null's +0.022, b_prox +0.363 [+0.326, +0.401], as printed.")

b_other +0.125 [+0.091, +0.158], null 97.5th percentile +0.022; b_prox +0.363 [+0.326, +0.401] -> CORE AND CULTURAL
Refitted: b_other +0.125 [+0.091, +0.158] above its null's +0.022, b_prox +0.363 [+0.326, +0.401], as printed.


## One person's blind check of the map (Table 14)

In [4]:
def kappa_nominal(a, b):
    """Cohen's kappa for nominal codes (shared/stats.py's kappa_nominal)."""
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    ca, cb = Counter(a), Counter(b)
    pe = sum(ca[l] * cb[l] for l in set(a) | set(b)) / n ** 2
    return (po - pe) / (1 - pe) if pe < 1 else float("nan")

codes = load_code_json("studies/identifiability/core-vs-culture/out/human-codes.json")
maps = load_code_json("studies/identifiability/core-vs-culture/out/unit-map.json")["maps"]
sample = load_code_json("studies/identifiability/core-vs-culture/out/human-sample.json")["items"]
M1, M2 = "openai/gpt-6-sol", "deepseek/deepseek-v4-pro"
rows = codes["codes"]
h = [r["code"] for r in rows]
m1 = [maps[M1][r["corpus"]][r["unit"]] for r in rows]
m2 = [maps[M2].get(r["corpus"], {}).get(r["unit"], "NONE") for r in rows]
corp = [r["corpus"] for r in rows]
domain = lambda c: c if c == "NONE" else c.split(".")[1]
k1, k2, k12 = kappa_nominal(h, m1), kappa_nominal(h, m2), kappa_nominal(m1, m2)
same_dom = np.mean([domain(x) == domain(y) for x, y in zip(h, m1)])
print(f"coder {codes['coder']}, {len(rows)} items coded of {len(sample)}: kappa with the primary map {k1:.3f}, "
      f"with the second {k2:.3f}; maps with each other {k12:.3f}; same domain {same_dom:.3f}")
assert (len(rows), len(sample) - len(rows)) == (87, 13)
assert (f"{k1:.3f}", f"{k2:.3f}", f"{k12:.3f}", f"{same_dom:.3f}") == ("0.606", "0.560", "0.710", "0.759")

# the interval: 2,000 resamples of items within corpus, with the study's seed and draw order
rng = np.random.default_rng(20261007)
H, B1, C = np.array(h), np.array(m1), np.array(corp)
groups = {c: np.where(C == c)[0] for c in sorted(set(corp))}
boots = []
for _ in range(2000):
    ix = np.concatenate([rng.choice(g, len(g)) for g in groups.values()])
    boots.append(kappa_nominal(list(H[ix]), list(B1[ix])))
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"kappa {k1:.3f} [{lo:.3f}, {hi:.3f}]; the registered bar needs >= 0.60 with a lower bound >= 0.40: "
      f"passes by {k1 - 0.60:.3f}")
assert (f"{lo:.3f}", f"{hi:.3f}", f"{k1 - 0.60:.3f}") == ("0.493", "0.709", "0.006") and k1 >= 0.60 and lo >= 0.40

PRINTED_T14 = {"assistments2012": (25, "0.620", "0.582", "0.706", "0.800"), "eedi": (24, "0.668", "0.563", "0.702", "0.750"),
               "junyi2020": (20, "0.525", "0.520", "0.581", "0.750"), "xes3g5m": (18, "0.464", "0.445", "0.825", "0.722")}
for c, want in PRINTED_T14.items():
    g = groups[c]
    a, b, d = [h[i] for i in g], [m1[i] for i in g], [m2[i] for i in g]
    got = (len(g), f"{kappa_nominal(a, b):.3f}", f"{kappa_nominal(a, d):.3f}", f"{kappa_nominal(b, d):.3f}",
           f"{np.mean([domain(x) == domain(y) for x, y in zip(a, b)]):.3f}")
    print(c, got)
    assert got == want, c

# the unregistered reading: the 13 blank items as "no cluster"
coded = {(r["corpus"], r["unit"]): r["code"] for r in rows}
hb = [coded.get((it["corpus"], it["unit"]), "NONE") for it in sample]
mb = [maps[M1][it["corpus"]][it["unit"]] for it in sample]
kb = kappa_nominal(hb, mb)
print(f"with the blanks read as no cluster: {kb:.3f} over {len(hb)} items")
assert (f"{kb:.3f}", len(hb)) == ("0.549", 100)
print("Table 14 and the kappas match the paper.")

coder human:david-porcaro, 87 items coded of 100: kappa with the primary map 0.606, with the second 0.560; maps with each other 0.710; same domain 0.759
kappa 0.606 [0.493, 0.709]; the registered bar needs >= 0.60 with a lower bound >= 0.40: passes by 0.006
assistments2012 (25, '0.620', '0.582', '0.706', '0.800')
eedi (24, '0.668', '0.563', '0.702', '0.750')
junyi2020 (20, '0.525', '0.520', '0.581', '0.750')
xes3g5m (18, '0.464', '0.445', '0.825', '0.722')
with the blanks read as no cluster: 0.549 over 100 items
Table 14 and the kappas match the paper.


## The pair-level scan on XES3G5M (Table 13)

In [5]:
scan = next(c for c in load_code_json("studies/identifiability/learning-structure/out/iita-pairscan.json")["corpora"]
            if c["corpus"] == "xes3g5m")
scored = [s for s in scan["sets"] if "t_obs" in s]
print(f"{'set':>3s} {'learners':>8s} {'min z':>7s} {'crit':>7s} {'p':>6s} isolated pairs")
pairs_total = 0
for s in scored:
    unordered = {frozenset((p["prereq"], p["dependant"])) for p in s["pairs"] if p["isolated"]}
    pairs_total += len(unordered)
    print(f"{s['set']:3d} {s['learners']:8,d} {s['t_obs']:7.2f} {s['critical_z']:7.2f} {s['p_maxT']:6.3f} "
          f"{'yes' if s['fires_isolated'] else 'no':8s} {len(unordered)}")
    assert s["fires_isolated"] == (s["t_obs"] < s["critical_z"] and bool(unordered))   # fires: past the critical value, isolated
PRINTED_T13 = [  # set: learners, most negative z, critical value, max-statistic p (the minus sign as printed)
    (3000, "−4.46", "−3.73", "0.002"), (1344, "−4.73", "−3.79", "0.002"), (1544, "−5.01", "−3.73", "0.004"),
    (605, "−2.71", "−3.78", "0.473"), (1165, "−4.98", "−3.90", "0.002"), (782, "−3.66", "−3.90", "0.044"),
    (705, "−3.68", "−3.69", "0.026"), (1256, "−5.08", "−3.80", "0.002"), (1136, "−5.01", "−3.77", "0.002"),
    (461, "−3.35", "−4.11", "0.108"), (737, "−4.01", "−3.93", "0.024"), (354, "−3.21", "−4.85", "0.321"),
    (467, "−4.25", "−4.18", "0.018"), (604, "−2.54", "−4.04", "0.695"), (278, "−6.01", "−5.20", "0.008")]
assert [(s["learners"], sg(s["t_obs"], 2), sg(s["critical_z"], 2), f"{s['p_maxT']:.3f}") for s in scored] == PRINTED_T13
fires = sum(s["fires_isolated"] for s in scored)
p_binom = sum(math.comb(len(scored), k) * 0.05 ** k * 0.95 ** (len(scored) - k) for k in range(fires, len(scored) + 1))
print(f"{fires} of {len(scored)} sets fire on {pairs_total} pairs; binomial p {p_binom:.1e} against a chance rate of 0.05")
assert (fires, len(scored), pairs_total, f"{p_binom:.1e}") == (8, 15, 12, "1.8e-07")
assert abs(p_binom - scan["binomial_p"]) < 1e-15 and scan["verdict"] == "PAIRWISE LOCAL DEPENDENCE BEYOND ONE TRAIT"
assert [s["set"] for s in scored if s["fires_isolated"]] == [0, 1, 2, 4, 7, 8, 10, 12]
s14 = next(s for s in scored if s["set"] == 14)
assert s14["fires_clustered"] and not s14["fires_isolated"]        # set 14 shows only clustered dependence
print("Table 13 matches: sets 0, 1, 2, 4, 7, 8, 10 and 12 fire, on 12 pairs.")

set learners   min z    crit      p isolated pairs
  0    3,000   -4.46   -3.73  0.002 yes      2
  1    1,344   -4.73   -3.79  0.002 yes      1
  2    1,544   -5.01   -3.73  0.004 yes      3
  3      605   -2.71   -3.78  0.473 no       0
  4    1,165   -4.98   -3.90  0.002 yes      1
  5      782   -3.66   -3.90  0.044 no       0
  6      705   -3.68   -3.69  0.026 no       0
  7    1,256   -5.08   -3.80  0.002 yes      1
  8    1,136   -5.01   -3.77  0.002 yes      2
  9      461   -3.35   -4.11  0.108 no       0
 10      737   -4.01   -3.93  0.024 yes      1
 11      354   -3.21   -4.85  0.321 no       0
 12      467   -4.25   -4.18  0.018 yes      1
 13      604   -2.54   -4.04  0.695 no       0
 14      278   -6.01   -5.20  0.008 no       0
8 of 15 sets fire on 12 pairs; binomial p 1.8e-07 against a chance rate of 0.05
Table 13 matches: sets 0, 1, 2, 4, 7, 8, 10 and 12 fire, on 12 pairs.


## The pairwise gap matrix and its parts (Table 12)

In [6]:
lr = load_code_json("studies/identifiability/latent-edge/out/lowrank.json")["rows"]
for corpus, want in (("xes3g5m", ("+0.522", "+0.849", ["+0.376", "+0.318", "+0.293", "+0.265"])),
                     ("ednet", ("+0.061", "+0.276", ["+0.026", "+0.022", "+0.020", "+0.016"]))):
    rs = sorted((r for r in lr if r["corpus"] == corpus), key=lambda r: r["deflated"])
    got = (sg(rs[0]["raw"]), sg(rs[0]["additive"]), [sg(r["residual"]) for r in rs])
    print(corpus, got, "nulls", [sg(r["residual_null"]) for r in rs])
    assert got == want, corpus
arms = [r for r in lr if r["deflated"] >= 1]
assert len(arms) == 6 and all(r["residual"] > r["residual_null"] for r in arms)   # all 6 arms with k >= 1 beat their null
print("Table 12 matches; all 6 arms with k >= 1 replicate above their null: PAIRWISE STRUCTURE.")

xes3g5m ('+0.522', '+0.849', ['+0.376', '+0.318', '+0.293', '+0.265']) nulls ['+0.002', '−0.001', '+0.002', '+0.002']
ednet ('+0.061', '+0.276', ['+0.026', '+0.022', '+0.020', '+0.016']) nulls ['−0.002', '−0.001', '+0.002', '−0.002']
Table 12 matches; all 6 arms with k >= 1 replicate above their null: PAIRWISE STRUCTURE.
